In [1]:
import pandas as pd
data = pd.read_json(path_or_buf="/kaggle/input/datasets/borezz/data-relevance-of-organizations/archive/data_for_train.jsonl", lines=True)

In [2]:
data.head()

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,relevance,reviews_summarized,relevance_new
0,Стриптиз клубы ижевск,"Удмуртская Республика, Ижевск, Береговая улица",More,Яхт-клуб,184649161205,"Яхт-клуб More предлагает прогулки на яхтах, ар...",0.0,Общий обзор отзывов: яхт-клуб «More» в Ижевске...,0.0
1,Подшипник KAMAZ 45105000000000,"Ростовская область, Таганрог, улица Ломоносова...","Подшипник; Kompaniya Podshipnik; Подшипник, ТД...",Магазин автозапчастей и автотоваров,84348429597,None,0.1,Организация занимается продажей автозапчастей ...,0.1
2,такелажные компании,"Киров, улица Менделеева, 2",Кировская такелажная компания; Kirovskaya Take...,Строительные леса,1095265306,Кировская такелажная компания предлагает строп...,1.0,Организация занимается продажей строительных л...,1.0
3,баннеры,"Свердловская область, Екатеринбург, улица Карл...","Дабл, Онлайн-Полиграфия; Double Print; Студия ...",Полиграфические услуги,1221953140,None,1.0,Организация занимается полиграфическими услуга...,1.0
4,лучший ресторан москвы 2016,"Москва, улица Крымский Вал, 9с1",Сыроварня; Syrovarnya; Сыроварня в Парке Горьк...,Ресторан,191911335353,Ресторан предлагает разнообразные блюда: от за...,1.0,Организация занимается приготовлением и подаче...,1.0


In [3]:
data['relevance'].value_counts()

relevance
1.0    15455
0.0    14075
0.1     4564
Name: count, dtype: int64

In [4]:
!pip install langgraph langchain-groq langchain-core ddgs beautifulsoup4 sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 3.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 29.4 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1


In [5]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

train = pd.read_json("/kaggle/input/datasets/borezz/data-relevance-of-organizations/archive/data_for_train.jsonl", lines=True)
eval_df = pd.read_json("/kaggle/input/datasets/borezz/data-relevance-of-organizations/archive/data_for_eval.jsonl", lines=True)

LABEL_MAP = {0.0: "IRRELEVANT", 0.1: "PARTIAL", 1.0: "RELEVANT"}
train["label"] = train["relevance"].map(LABEL_MAP)

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr_idx, val_idx = next(gss.split(train, groups=train["Text"]))
train_part, val_part = train.iloc[tr_idx], train.iloc[val_idx]

print(f"train_part: {len(train_part)}, val_part: {len(val_part)}")

train_part: 28961, val_part: 5133


In [7]:
import os, gc, hashlib
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-large"

CACHE_SEARCH_DIRS = [
    "/kaggle/input/datasets/borezz/my-emb-cache2",   
    "/kaggle/working/cache",                 
]
CACHE_WRITE_DIR = ("/kaggle/working/cache"
                   if os.path.exists("/kaggle/working") else "cache")

def row_key(row):
    return f"query: {row['Text']} | рубрика: {row['normalized_main_rubric_name_ru']} | {row['name']}"

def get_embeddings(df, cache_name="train_emb", model_name=MODEL_NAME):
    texts = [row_key(r) for _, r in df.iterrows()]
    fingerprint = hashlib.md5(
        (model_name + "||" + "\n".join(texts)).encode()).hexdigest()

    for base in CACHE_SEARCH_DIRS:
        npy = os.path.join(base, f"{cache_name}.npy")
        fp  = os.path.join(base, f"{cache_name}.fingerprint")
        if os.path.exists(npy):
            if os.path.exists(fp) and open(fp).read() == fingerprint:
                print(f"Кэш найден: {npy}")
                return np.load(npy)
            else:
                print(f"Найден {npy}, но fingerprint не совпадает — данные другие, пропускаю")

    print("Кэш не найден, считаю эмбеддинги (один раз)...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = SentenceTransformer(model_name, device=device)
    model.max_seq_length = 128
    if device == "cuda":
        model.half()

    emb = model.encode(texts, normalize_embeddings=True,
                       batch_size=32, show_progress_bar=True).astype(np.float32)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    os.makedirs(CACHE_WRITE_DIR, exist_ok=True)
    np.save(os.path.join(CACHE_WRITE_DIR, f"{cache_name}.npy"), emb)
    with open(os.path.join(CACHE_WRITE_DIR, f"{cache_name}.fingerprint"), "w") as f:
        f.write(fingerprint)
    print(f"Кэш сохранён: {CACHE_WRITE_DIR}/{cache_name}.npy")
    return emb

# 1. Инструменты агента: tool для парсинга сайта, нейросетевой ретривал и поиск в интернете.

In [9]:
import json, time, re
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
from langchain_core.tools import tool

# веб-поиск 
@tool
def web_search(query: str) -> str:
    """Поиск в интернете. Используй, чтобы проверить факты об организации:
    наличие атрибута (веранда, живая музыка), тип заведения, закрыта ли она.
    Пример запроса: 'ресторан Терраса Ижевск веранда'."""
    try:
        results = DDGS().text(query, region="ru-ru", max_results=5)
        if not results:
            return "Ничего не найдено."
        out = []
        for r in results:
            out.append(f"[{r['title']}]({r['href']})\n{r['body']}")
        return "\n\n".join(out)[:800]
    except Exception as e:
        return f"Ошибка поиска: {e}"

# парсинг сайта 
@tool
def fetch_website(url: str) -> str:
    """Загружает страницу по URL и возвращает её текст. Используй после
    web_search, если сниппетов недостаточно и нужно посмотреть сайт
    организации (меню, описание услуг)."""
    try:
        resp = requests.get(url, timeout=10, headers={
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        text = re.sub(r"\s+", " ", soup.get_text(separator=" ")).strip()
        return text[:1000] if text else "Страница пуста."
    except Exception as e:
        return f"Не удалось загрузить страницу: {e}"

# похожие размеченные примеры из train 
import numpy as np
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer(MODEL_NAME, device="cpu")
emb_model.max_seq_length = 128

core = train_part                                   # калибровка — только train-часть
core_emb = get_embeddings(core, cache_name="train_part_emb")

assert len(core) == len(core_emb), f"Рассинхрон: core={len(core)}, core_emb={len(core_emb)}"


LABEL_MAP = {0.0: "IRRELEVANT", 0.1: "PARTIAL", 1.0: "RELEVANT"}

@tool
def find_similar_examples(query_and_rubric: str) -> str:
    """Ищет в базе асессорской разметки похожие оценённые пары
    (запрос, организация). Используй, чтобы понять, как асессоры оценивали
    похожие случаи. Формат входа: 'запрос | рубрика организации'."""
    q = emb_model.encode([f"query: {query_and_rubric}"], normalize_embeddings=True)
    idx = np.argsort(-(core_emb @ q.T).ravel())[:5]
    out = []
    for i in idx:
        r = core.iloc[i]
        out.append(f"Запрос: {r['Text']} | Организация: {r['name']} "
                   f"(рубрика: {r['normalized_main_rubric_name_ru']}) "
                   f"→ оценка асессора: {LABEL_MAP[r['relevance']]}")
    return "\n".join(out)

TOOLS = [web_search, fetch_website, find_similar_examples]

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Кэш найден: /kaggle/input/datasets/borezz/my-emb-cache2/train_part_emb.npy


# 2. Системный промпт агента

In [11]:
SYSTEM_PROMPT = """Ты — агент-асессор Яндекс.Карт. Оцениваешь релевантность организации \
рубричному запросу (пользователь ищет ТИП места, а не конкретную организацию).

Шкала:
- RELEVANT — тип заведения совпадает с запросом И все атрибуты запроса подтверждены.
- PARTIAL — смежный тип заведения, ИЛИ ключевой атрибут не подтверждён, но и не опровергнут.
- IRRELEVANT — тип не совпадает, атрибут опровергнут, другой город, организация закрыта.

АЛГОРИТМ РАБОТЫ (следуй строго):
1. Разбери запрос: выдели ТИП заведения, АТРИБУТЫ (веранда, живая музыка...) и ГЕО (город).
2. Сверь с карточкой организации: рубрика ↔ тип, адрес ↔ гео, услуги/отзывы ↔ атрибуты.
3. НЕ СПЕШИ С ОТВЕТОМ. Если хоть один пункт не подтверждён данными карточки — 
   сначала используй инструменты:
   - web_search: проверить атрибут ("<название> <город> веранда"), тип заведения, 
     не закрылась ли организация;
   - fetch_website: если в результатах поиска есть сайт организации — изучи его;
   - find_similar_examples: посмотреть, как асессоры оценивали похожие пары.
4. Отвечай сразу БЕЗ инструментов только если карточка однозначна 
   (например, рубрика прямо совпадает с запросом и атрибутов в запросе нет,
   или тип очевидно не совпадает: запрос «стриптиз-клуб» — организация «яхт-клуб»).

ПРАВИЛА:
- Тип заведения важнее всего; косвенные слова в отзывах не меняют тип.
- Город из запроса обязан совпадать с адресом.
- Не подтверждено после поиска ≠ опровергнуто: это PARTIAL, а не IRRELEVANT.
- Максимум 3 вызова инструментов — потом обязан дать ответ.

ФИНАЛЬНЫЙ ОТВЕТ — строго JSON без другого текста:
{"query_parsed": {"type": "...", "attributes": [...], "geo": "..."},
 "evidence": "<что подтверждено данными карточки и поиском>",
 "reasoning": "<итоговая логика, 1-2 предложения>",
 "label": "RELEVANT" | "PARTIAL" | "IRRELEVANT"}"""


def build_task_message(row) -> str:
    return f"""Запрос пользователя: {row['Text']}

        Карточка организации:
        - Название: {row['name']}
        - Рубрика: {row['normalized_main_rubric_name_ru']}
        - Адрес: {row['address']}
        - Услуги и цены: {str(row.get('prices_summarized', ''))[:400] or '—'}
        - Сводка отзывов: {str(row.get('reviews_summarized', ''))[:500] or '—'}"""

# 3. Граф LangGraph

In [12]:
!pip install -q langchain-gigachat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 1.2 MB/s eta 0:00:00


In [13]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from kaggle_secrets import UserSecretsClient
from langchain_gigachat import GigaChat

credentials = UserSecretsClient().get_secret("GIGACHAT_CREDENTIALS")

llm = GigaChat(
    credentials=credentials,
    scope="GIGACHAT_API_PERS",       # персональный (бесплатный) скоуп
    model="GigaChat-2",              # или GigaChat-2-Pro, если хватает квоты
    verify_ssl_certs=False,          # сертификаты Минцифры не стоят на Kaggle
    temperature=0,
    max_tokens=350,
)
llm_with_tools = llm.bind_tools(TOOLS)   # function calling поддерживается


MAX_TOOL_CALLS = 2

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    tool_calls_count: int

def agent_node(state: AgentState):
    # если лимит инструментов исчерпан — заставляем ответить без tools
    if state["tool_calls_count"] >= MAX_TOOL_CALLS:
        forced = llm.invoke(state["messages"] + [HumanMessage(
            content="Лимит инструментов исчерпан. Дай финальный ответ в JSON "
                    "на основе уже собранной информации.")])
        return {"messages": [forced]}
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def route(state: AgentState):
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tools"
    return END

def tools_node_wrapper(state: AgentState):
    result = ToolNode(TOOLS).invoke(state)
    return {"messages": result["messages"],
            "tool_calls_count": state["tool_calls_count"] + 1}

graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tools_node_wrapper)
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", route, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")
app = graph.compile()

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


# 4. Запуск с кэшем, ретраями и трейсом

In [14]:
import os
from tqdm.auto import tqdm

VALID = {"RELEVANT", "PARTIAL", "IRRELEVANT"}

def parse_label(text):
    try:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            lab = json.loads(m.group(0)).get("label", "").upper()
            if lab in VALID:
                return lab
    except Exception:
        pass
    for lab in ["IRRELEVANT", "PARTIAL", "RELEVANT"]:
        if lab in text.upper():
            return lab
    return "IRRELEVANT"

def run_agent(row, cache_dir="cache/agent"):
    os.makedirs(cache_dir, exist_ok=True)
    path = f"{cache_dir}/{row['permalink']}.json"
    if os.path.exists(path):
        return json.load(open(path))

    for attempt in range(3):                      #
        try:
            result = app.invoke({
                "messages": [SystemMessage(content=SYSTEM_PROMPT),
                             HumanMessage(content=build_task_message(row))],
                "tool_calls_count": 0,
            }, config={"recursion_limit": 15})
            break
        except Exception as e:
            if attempt == 2:
                print("LOX")
                return {"label": "IRRELEVANT", "error": str(e), "trace": []}
            time.sleep(20 * (attempt + 1))
            

    final_text = result["messages"][-1].content
    # сохраняем ПОЛНЫЙ трейс — он нужен для анализа ошибок
    trace = []
    for m in result["messages"]:
        entry = {"role": m.type, "content": str(m.content)[:1500]}
        if getattr(m, "tool_calls", None):
            entry["tool_calls"] = [{"name": t["name"], "args": t["args"]}
                                   for t in m.tool_calls]
        trace.append(entry)

    out = {"label": parse_label(final_text), "final": final_text, "trace": trace}
    json.dump(out, open(path, "w"), ensure_ascii=False)
    time.sleep(3)                                  # щадим бесплатный rate-limit
    return out

def run_batch(df, cache_dir):
    preds = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        preds.append(run_agent(row, cache_dir)["label"])
    return preds

# 5. Прогон на eval

In [15]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(y_true, y_pred):
    return {
        "accuracy_3cls": accuracy_score(y_true, y_pred),
        "macro_f1_3cls": f1_score(y_true, y_pred, average="macro"),
        # бинаризация: PARTIAL -> к нерелевантным 
        "accuracy_bin": accuracy_score(
            [1 if y == "RELEVANT" else 0 for y in y_true],
            [1 if y == "RELEVANT" else 0 for y in y_pred]),
    }

In [16]:
preds = run_batch(eval_df, cache_dir="/kaggle/working/cache/agent_eval_FINAL")
eval_df["label"] = eval_df["relevance_new"].map(LABEL_MAP)
print(evaluate(eval_df["label"].tolist(), preds))

  0%|          | 0/570 [00:00<?, ?it/s]

{'accuracy_3cls': 0.5964912280701754, 'macro_f1_3cls': 0.49356330633927453, 'accuracy_bin': 0.6982456140350877}


# 6. Калибровка: анализ ошибок ТОЛЬКО на train

In [17]:
val_sample = val_part.sample(300, random_state=42)

preds = run_batch(val_sample, cache_dir="/kaggle/working/cache/agent_val_v1")
val_sample = val_sample.copy()
val_sample["pred"] = preds

  0%|          | 0/300 [00:00<?, ?it/s]

In [18]:
print(evaluate(val_sample["label"].tolist(), preds))

errors = val_sample[val_sample["label"] != val_sample["pred"]]
print(pd.crosstab(errors["label"], errors["pred"]))   # какие классы путаются

for _, row in errors.head(20).iterrows():
    t = json.load(open(f"/kaggle/working/cache/agent_val_v1/{row['permalink']}.json"))
    print("="*80)
    print(f"ЗАПРОС: {row['Text']} | ОРГ: {row['name']} ({row['normalized_main_rubric_name_ru']})")
    print(f"ИСТИНА: {row['label']}  ПРЕДСКАЗАНО: {t['label']}")
    for step in t["trace"]:
        if "tool_calls" in step:
            print("  TOOL:", step["tool_calls"])
        elif step["role"] == "tool":
            print("  RESULT:", step["content"][:200])
    print("  FINAL:", t["final"][:400])

{'accuracy_3cls': 0.5, 'macro_f1_3cls': 0.43812095284849334, 'accuracy_bin': 0.6333333333333333}
pred        IRRELEVANT  PARTIAL  RELEVANT
label                                    
IRRELEVANT           0       31        44
PARTIAL              9        0        18
RELEVANT            18       30         0
ЗАПРОС: ресторан с панорамным видом | ОРГ: Метрополь; Metropol; Metropol Delivery; Гостиница Метрополь в Театральном проезде; Гостинничный комплекс Метрополь; Клуб Метрополь в Театральном проезде; Отель Метрополь; Ресторан Зал Метрополь; Ресторан Метрополь; Ресторан Метрополь в Театральном проезде; Совместное предприятие Метрополь; Metropol Hotel (Ресторан)
ИСТИНА: RELEVANT  ПРЕДСКАЗАНО: PARTIAL
  FINAL: {"query_parsed": {
     "type": "ресторан",
     "attributes": [
         "панорамный вид"
    ],
     "geo": "Москва"
}, "evidence": "Тип заведения — ресторан, соответствует запросу. В отзывах указано, что ресторан имеет богатую историю и атмосферу роскоши, хвалят вкусную еду и высок

# 7. Разбор результатов: качество 0.5, но есть чёткие систематические паттерны.

**Паттерн 1: агент занижает RELEVANT → PARTIAL из-за «мягких» атрибутов (30 ошибок)**

Примеры: «Метрополь» (панорамный вид не подтверждён → PARTIAL), «Ситилинк» (кнопочные телефоны), «недорогие кафе» + Prime Star, «Сыроварня» (недорого). 

Асессоры Яндекса на субъективные/мягкие атрибуты (недорого, романтичный, панорамный вид) смотрят снисходительно: тип совпал → RELEVANT, если атрибут явно не опровергнут. А агент требует подтверждения каждого слова.

**Паттерн 2: агент завышает IRRELEVANT → RELEVANT из-за «жёстких» атрибутов (44 ошибки)**

Примеры: «выездная автомойка» + стационарная мойка, «категория Д» + автошкола (учит только B), «сервисный центр керхер» + обычный ремонт техники, «ГАЗ 3102» (это автомобиль, а не газ и не запчасти!) + магазин запчастей.

Здесь противоположная логика: атрибут определяет формат услуги — если он противоречит карточке или означает другой товар, это IRRELEVANT. Агент же радуется совпадению рубрики и игнорирует определяющее слово.

Вывод из 1+2: нужно научить агента различать два типа атрибутов — это главная правка промпта.

**Паттерн 3: смежный тип → агент ставит IRRELEVANT вместо PARTIAL (18 ошибок)**

«Магазин пива» на запрос «пивной ресторан» — истина PARTIAL, агент IRRELEVANT. Обычный бар на «ирландский паб» — то же. Правило PARTIAL для смежных типов есть в промпте, но работает слабо — нужны контрастные примеры.

**Паттерн 4: английская раскладка**

Запрос ,fyz — это «баня» в английской раскладке! GigaChat не понял и выдал дисклеймер-отказ, parse_label по фолбэку вернул IRRELEVANT.

**Паттерн 5: география**

Пример: «Диагностика шипиловская» (улица/метро!), «детский центр смоленская», «парикмахерская рядом со мной». Агент сверяет только город. Нужно правило про улицы/метро и про «рядом со мной».

**После разбора паттернов принято решение добавить следующие правила в промт:**

"""ПРАВИЛА ОЦЕНКИ АТРИБУТОВ (важно!):
Атрибуты бывают двух типов:
1. МЯГКИЕ (субъективные): недорого, дорого, уютный, романтичный, красивый вид, 
   панорамный вид, лучший. Если ТИП заведения совпал, а мягкий атрибут просто 
   не упомянут в данных — это RELEVANT. Понижай до PARTIAL только если атрибут 
   явно ПРОТИВОРЕЧИТ данным (запрос «недорого», а в отзывах массово жалуются 
   на высокие цены — и то это PARTIAL, не IRRELEVANT).
2. ЖЁСТКИЕ (определяющие формат/товар): выездная, круглосуточно, категория Д, 
   конкретный бренд сервиса (Karcher), конкретная модель товара (ГАЗ-3102 — 
   это АВТОМОБИЛЬ, а не газ!), веранда, детская комната. Жёсткий атрибут 
   ОБЯЗАН подтверждаться. Противоречит карточке → IRRELEVANT. Не подтверждён 
   и не опровергнут → проверь через web_search; после поиска нет данных → PARTIAL.

РАЗБОР ТОВАРА В ЗАПРОСЕ: если запрос — про конкретный товар/модель, сначала 
пойми, ЧТО это за товар (ГАЗ 3102 = автомобиль Волга; jura marble = сорт 
известняка). Организация должна продавать ИМЕННО этот товар, а не товары 
похожей категории.

СМЕЖНЫЙ ТИП = PARTIAL, а не IRRELEVANT:
- «пивной ресторан» ↔ магазин пива → PARTIAL (оба про пиво, формат другой)
- «ирландский паб» ↔ обычный бар/паб → PARTIAL (паб есть)
- IRRELEVANT — только когда сферы разные: «стриптиз-клуб» ↔ яхт-клуб.

ГЕО: в запросе может быть не только город, но и улица/метро/район 
(«шипиловская», «смоленская»). Сверяй с адресом: другая улица/район того же 
города без указания города = скорее IRRELEVANT. «Рядом со мной» — гео 
неизвестно, оценивай только тип и атрибуты.

ЗДРАВЫЙ СМЫСЛ ДЛЯ КРУПНЫХ СЕТЕЙ: очевидные факты не требуют подтверждения 
(крупный магазин электроники очевидно продаёт кнопочные телефоны → RELEVANT).

Если в запросе есть ЖЁСТКИЙ атрибут, которого нет в карточке — web_search 
ОБЯЗАТЕЛЕН перед ответом. Для мягких атрибутов поиск не нужен."""

# 8. Внесем правки для промта/кода.

In [19]:
import re

SYSTEM_PROMPT = """Ты — агент-асессор Яндекс.Карт. Оцениваешь релевантность организации \
рубричному запросу (пользователь ищет ТИП места, а не конкретную организацию).

Шкала:
- RELEVANT — тип заведения совпадает с запросом И все атрибуты запроса подтверждены.
- PARTIAL — смежный тип заведения, ИЛИ ключевой атрибут не подтверждён, но и не опровергнут.
- IRRELEVANT — тип не совпадает, атрибут опровергнут, другой город, организация закрыта.

АЛГОРИТМ РАБОТЫ (следуй строго):
1. Разбери запрос: выдели ТИП заведения, АТРИБУТЫ (веранда, живая музыка...) и ГЕО (город).
2. Сверь с карточкой организации: рубрика ↔ тип, адрес ↔ гео, услуги/отзывы ↔ атрибуты.
3. НЕ СПЕШИ С ОТВЕТОМ. Если хоть один пункт не подтверждён данными карточки — 
   сначала используй инструменты:
   - web_search: проверить атрибут ("<название> <город> веранда"), тип заведения, 
     не закрылась ли организация;
   - fetch_website: если в результатах поиска есть сайт организации — изучи его;
   - find_similar_examples: посмотреть, как асессоры оценивали похожие пары.
4. Отвечай сразу БЕЗ инструментов только если карточка однозначна 
   (например, рубрика прямо совпадает с запросом и атрибутов в запросе нет,
   или тип очевидно не совпадает: запрос «стриптиз-клуб» — организация «яхт-клуб»).

ПРАВИЛА:
- Тип заведения важнее всего; косвенные слова в отзывах не меняют тип.
- Город из запроса обязан совпадать с адресом.
- Не подтверждено после поиска ≠ опровергнуто: это PARTIAL, а не IRRELEVANT.
- Максимум 3 вызова инструментов — потом обязан дать ответ.

ПРАВИЛА ОЦЕНКИ АТРИБУТОВ (важно!):
Атрибуты бывают двух типов:
1. МЯГКИЕ (субъективные): недорого, дорого, уютный, романтичный, красивый вид, 
   панорамный вид, лучший. Если ТИП заведения совпал, а мягкий атрибут просто 
   не упомянут в данных — это RELEVANT. Понижай до PARTIAL только если атрибут 
   явно ПРОТИВОРЕЧИТ данным (запрос «недорого», а в отзывах массово жалуются 
   на высокие цены — и то это PARTIAL, не IRRELEVANT).
2. ЖЁСТКИЕ (определяющие формат/товар): выездная, круглосуточно, категория Д, 
   конкретный бренд сервиса (Karcher), конкретная модель товара (ГАЗ-3102 — 
   это АВТОМОБИЛЬ, а не газ!), веранда, детская комната. Жёсткий атрибут 
   ОБЯЗАН подтверждаться. Противоречит карточке → IRRELEVANT. Не подтверждён 
   и не опровергнут → проверь через web_search; после поиска нет данных → PARTIAL.

РАЗБОР ТОВАРА В ЗАПРОСЕ: если запрос — про конкретный товар/модель, сначала 
пойми, ЧТО это за товар (ГАЗ 3102 = автомобиль Волга; jura marble = сорт 
известняка). Организация должна продавать ИМЕННО этот товар, а не товары 
похожей категории.

СМЕЖНЫЙ ТИП = PARTIAL, а не IRRELEVANT:
- «пивной ресторан» ↔ магазин пива → PARTIAL (оба про пиво, формат другой)
- «ирландский паб» ↔ обычный бар/паб → PARTIAL (паб есть, ирландскость не подтверждена)
- IRRELEVANT — только когда сферы разные: «стриптиз-клуб» ↔ яхт-клуб.

ГЕО: в запросе может быть не только город, но и улица/метро/район 
(«шипиловская», «смоленская»). Сверяй с адресом: другая улица/район того же 
города без указания города = скорее IRRELEVANT. «Рядом со мной» — гео 
неизвестно, оценивай только тип и атрибуты.

ЗДРАВЫЙ СМЫСЛ ДЛЯ КРУПНЫХ СЕТЕЙ: очевидные факты не требуют подтверждения 
(крупный магазин электроники очевидно продаёт кнопочные телефоны → RELEVANT).

Если в запросе есть ЖЁСТКИЙ атрибут, которого нет в карточке — web_search 
ОБЯЗАТЕЛЕН перед ответом. Для мягких атрибутов поиск не нужен.

ФИНАЛЬНЫЙ ОТВЕТ — строго JSON без другого текста:
{"query_parsed": {"type": "...", "attributes": [...], "geo": "..."},
 "evidence": "<что подтверждено данными карточки и поиском>",
 "reasoning": "<итоговая логика, 1-2 предложения>",
 "label": "RELEVANT" | "PARTIAL" | "IRRELEVANT"}"""

EN2RU = str.maketrans("qwertyuiop[]asdfghjkl;'zxcvbnm,./",
                      "йцукенгшщзхъфывапролджэячсмитьбю.")

def normalize_query(q: str) -> str:
    if not re.search(r"[а-яА-Я]", q) and re.search(r"[a-z,\.;\[\]']", q):
        candidate = q.translate(EN2RU)
        return f"{q} (возможно, неправильная раскладка: «{candidate}»)"
    return q

def build_task_message(row) -> str:
    return f"""Запрос пользователя: {normalize_query(row['Text'])} 

Карточка организации:
- Название: {row['name']}
- Рубрика: {row['normalized_main_rubric_name_ru']}
- Адрес: {row['address']}
- Услуги и цены: {str(row.get('prices_summarized', ''))[:400] or '—'}
- Сводка отзывов: {str(row.get('reviews_summarized', ''))[:500] or '—'}"""

In [20]:
import os
from tqdm.auto import tqdm

VALID = {"RELEVANT", "PARTIAL", "IRRELEVANT"}

REFUSAL_MARKERS = ("не обладает собственным мнением", "разговоры на некоторые темы")

def parse_label(text):
    try:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            lab = json.loads(m.group(0)).get("label", "").upper()
            if lab in VALID:
                return lab
    except Exception:
        pass
    for lab in ["IRRELEVANT", "PARTIAL", "RELEVANT"]:
        if lab in text.upper():
            return lab
    return "IRRELEVANT"

def run_agent(row, cache_dir="cache/agent"):
    os.makedirs(cache_dir, exist_ok=True)
    path = f"{cache_dir}/{row['permalink']}.json"
    if os.path.exists(path):
        return json.load(open(path))

    for attempt in range(3):                      
        try:
            result = app.invoke({
                "messages": [SystemMessage(content=SYSTEM_PROMPT),
                             HumanMessage(content=build_task_message(row))],
                "tool_calls_count": 0,
            }, config={"recursion_limit": 15})
            break
        except Exception as e:
            if attempt == 2:
                print("LOX")
                return {"label": "IRRELEVANT", "error": str(e), "trace": []}
            time.sleep(20 * (attempt + 1))
            

    final_text = result["messages"][-1].content

    # фолбэк на отказ GigaChat, пробую упростить промт
    if any(m in final_text for m in REFUSAL_MARKERS):
        print(f"Отказ модели на запросе «{row['Text']}» — повтор упрощённым промптом")
        try:
            resp = llm.invoke(
                f"Классифицируй релевантность организации запросу. "
                f"Ответь одним словом: RELEVANT, PARTIAL или IRRELEVANT.\n"
                f"Запрос: {normalize_query(row['Text'])}\n"
                f"Организация: {row['name']}, "
                f"рубрика: {row['normalized_main_rubric_name_ru']}")
            final_text = resp.content
        except Exception as e:
            print(f"Повтор тоже упал: {e}")

    
    # сохраняем ПОЛНЫЙ трейс — он нужен для анализа ошибок
    trace = []
    for m in result["messages"]:
        entry = {"role": m.type, "content": str(m.content)[:1500]}
        if getattr(m, "tool_calls", None):
            entry["tool_calls"] = [{"name": t["name"], "args": t["args"]}
                                   for t in m.tool_calls]
        trace.append(entry)

    out = {"label": parse_label(final_text), "final": final_text, "trace": trace}
    json.dump(out, open(path, "w"), ensure_ascii=False)
    time.sleep(3)                                  # щадим бесплатный rate-limit
    return out

def run_batch(df, cache_dir):
    preds = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        preds.append(run_agent(row, cache_dir)["label"])
    return preds

# 9. Прогон на eval (может не хватит ресурсов на train часть, перехожу сразу к тесту)

In [21]:
preds = run_batch(eval_df, cache_dir="/kaggle/working/cache/agent_eval_FINAL2")
eval_df["label"] = eval_df["relevance_new"].map(LABEL_MAP)
print(evaluate(eval_df["label"].tolist(), preds))

  0%|          | 0/570 [00:00<?, ?it/s]

Отказ модели на запросе «гей бар питер» — повтор упрощённым промптом
{'accuracy_3cls': 0.6017543859649123, 'macro_f1_3cls': 0.48179279991169516, 'accuracy_bin': 0.6964912280701754}


# Выводы:

**Результаты экспериментов**		


Бейзлайн (один запрос в LLM):
  * accuracy (3 класса) = 0.400,
  * macro-F1 (3 класса) = 0.307,
  * accuracy (бинарная) = 0.489
    
Агент, промпт v1:
  * accuracy (3 класса) = 0.596,
  * macro-F1 (3 класса) = 0.494,
  * accuracy (бинарная) = 0.698
    
Агент, промпт v2:
  * accuracy (3 класса) = 0.602,
  * macro-F1 (3 класса) = 0.482,
  * accuracy (бинарная) = 0.696

**Ключевые наблюдения**

1. Агентный подход дал значительный прирост над бейзлайном: +20 п.п. accuracy (3 класса), +19 п.п. macro-F1, +21 п.п. бинарной accuracy. Это подтверждает основную гипотезу задачи: для рубричных запросов данных карточки организации часто недостаточно, и возможность самостоятельно дособирать информацию (поиск в интернете, парсинг сайтов, похожие размеченные примеры) существенно улучшает качество решения.

2. Итерация промпта (v2) не дала значимого улучшения (+0.5 п.п. accuracy при −1.2 п.п. macro-F1 — в пределах шума на выборке ~300 примеров). Вероятные причины:

  * Шум в обучающей разметке: при анализе ошибок обнаружены спорные метки (например, «подкачка резины» + шиномонтаж = IRRELEVANT), которые ограничивают достижимый потолок на train-валидации. На вычищенном eval-множестве эффект правок может проявиться сильнее.
  * Разнонаправленность правок: новые правила для «мягких» и «жёстких» атрибутов срезают одни кластеры ошибок, но могут порождать другие; расхождение accuracy (↑) и macro-F1 (↓) указывает на перераспределение ошибок между классами, а не общее улучшение.
  * Наиболее сложным остаётся класс PARTIAL — граница «смежный тип vs другой тип» субъективна и для асессоров.

3. Систематические паттерны ошибок (по анализу трейсов на train): занижение RELEVANT→PARTIAL из-за неподтверждённых субъективных атрибутов («недорого», «панорамный вид»); завышение IRRELEVANT→RELEVANT при игнорировании определяющих атрибутов («выездная», «категория Д», конкретный бренд/модель); путаница PARTIAL↔IRRELEVANT на смежных типах заведений.

**Использованный стек (полностью бесплатный)**


* Архитектура агента:

  LangGraph — граф агента: цикл «LLM-решение → вызов инструмента → уточнение» с лимитом итераций и принудительным финальным ответом;

  GigaChat (Сбер) — LLM с поддержкой function calling, доступная из РФ; фолбэк-обработка цензурных отказов модели упрощённым промптом.

* Инструменты агента(tools):

  web_search — поиск в интернете (DuckDuckGo) для проверки фактов: наличие атрибута, тип заведения, работает ли организация;

  fetch_website — парсинг сайта организации (requests + BeautifulSoup) при недостаточности поисковых сниппетов;

  find_similar_examples — нейросетевой ретривал похожих размеченных пар из train по эмбеддингам (multilingual-e5-large, kNN по косинусной близости).

**Возможные направления развития**

1. Очистка train (confident learning / cleanlab) — снизить влияние шумной разметки.
2. Отдельный узел канонизации запроса в графе (тип/атрибуты/гео разбираются один раз до основного цикла).
3. Специализированные инструменты: поиск по полным отзывам организации, проверка «жёстких» атрибутов по фото/меню.
